In [2]:
import os
from belgiumts_lib import parse_poles
import json
from ultralytics import YOLO
import cv2 as cv

In [2]:
classes_signs = set()
classes_bboxes = set()

poles_collection = {}

for gt_file in os.listdir('./data/belgiumts/sequences/'):
    if not gt_file.endswith('_GT.txt'):
        continue
    gt_path = os.path.join('./data/belgiumts/sequences/', gt_file)
    poles = parse_poles(gt_path)
    poles_collection[gt_file] = poles
    for pole in poles:
        for sign in pole.signs:
            classes_signs.add(sign.type.lower())

        for bbox in pole.bboxes:
            classes_bboxes.add(bbox.type.lower())

In [3]:
camera_1_signs = {}
for pole in poles_collection['sequence2_GT.txt']:
    for type in [bbox.type for bbox in pole.bboxes if bbox.camera == 1]:
        camera_1_signs[type] = camera_1_signs.get(type, 0) + 1

# for x in sorted(camera_1_signs.keys()):
#     print(x + ': ', mapped_classes_from_belgiumts_to_dfg.get(x.lower(), []))

In [42]:
for x in sorted(camera_1_signs.keys()):
    print(x + ': ')

50m: 
7,5t_uitgezonderd_laden_en_lossen: 
7,5t_uitgezonderd_laden_lossen: 
A13: 
A23: 
A23_geel: 
A7C: 
B1: 
B17: 
B5: 
B9: 
C31LEFT: 
C31RIGHT: 
C43: 
D1b_rechts: 
D1b_rechts_onder: 
D7: 
D9: 
E1: 
E3: 
E9a: 
E9a_miva: 
E9b: 
F13: 
F17: 
F19: 
F34A: 
F43: 
F4a: 
F50bis: 
F97B: 
MAX3,5t: 
P7t_einde: 
STOP_150m: 
begin: 
blabla: 
e0c: 
einde: 
groene_fiets: 
lang: 
max3,5t: 
oranje_wandel: 
zone_P_3,5t_max: 
zone_verboden_camion_blabla: 


In [41]:
defined_signs = set()
with open('./data/belgiumts/sequences/list_defined_TS.txt', 'r') as f:
    lines = f.readlines()

for line in lines:
    if line.strip().endswith('.png'):
        defined_signs.add(line.strip().split('.')[0].lower())

In [45]:
for x in sorted(classes_signs - defined_signs):
    print(x)

25m
3,5t
30m
50m
7,5t_uitgezonderd_laden_en_lossen
7,5t_uitgezonderd_laden_lossen
7,5tuitgezonderd_laden_en_lossen
700m
90m
a23_geel
bebouwde_kom_50km_radar_controle
bewoners_politievoertuigen
blabla
d1b_rechts
d1b_rechts_onder
d1b_schuin_rechts
db1_schuin_rechts
e9b_disk
einde_zone_idunno
f29_f23a
f29_f23a_f23b
f29_links
f29_links_f23a
f30
f34a_links
f37_links
f50bis variant
f59_links
fiets_wandel_routes
fietsers_rijbaan
gevaar_fiets
groen_fiets
groene_fiets
herhaling
hulpdiensten
laden_lossen
mag_weg
max3,5t
meter
oranje_wandel
oversteekplaats
p7t_einde
plaatselijk
school
stop_150m
typell
uigezonderd_plaatselijk_verkeer
uitgez_bus
uitgez_hulpdiensten
uitgezonderd_blabla
x10
x11
x2
x31
x31_buiten
x32
x32_buiten
x33
x3l
x3r
zeshoekige_fiets
zone7,5tverboden_combo_p3,5tmax
zone_einde_p_blabla
zone_idunno
zone_p_3,5t_max
zone_verboden_camion_blabla


In [3]:
with open('./data/belgiumts/sequences_jpg/seq01_01_track.json', 'r') as f:
    annotations = json.load(f)

In [14]:
annotations_by_img = {}
for ann in annotations:
    image = ann['image'].replace('.jp2', '.jpg')
    if image not in annotations_by_img:
        annotations_by_img[image] = []
    annotations_by_img[image].append(ann)

In [22]:
label_studio_anns = []
id = 1
# for img, anns in annotations_by_img.items():
for img in sorted(os.listdir('./data/belgiumts/sequences_jpg/Seq01/01/')):
    anns = annotations_by_img.get(img, [])
    result = []
    for ann in anns:
        x1 = ann['bbox'][0] - ann['bbox'][2] / 2
        y1 = ann['bbox'][1] - ann['bbox'][3] / 2
        result.append({
            'id': f'result{id}',
            'type': 'rectanglelabels',
            'from_name': 'label',
            'to_name': 'image',
            'original_width': 1628,
            'original_height': 1236,
            'image_rotation': 0,
            'value': {
                'rotation': 0,
                'x': x1 / 1628 * 100,
                'y': y1 / 1236 * 100,
                'width': ann['bbox'][2] / 1628 * 100,
                'height': ann['bbox'][3] / 1236 * 100,
                'rectanglelabels': [ann['class']]
            }
        })
        id += 1
    label_studio_path = '/data/local-files/?d=home/viktor/coding/nastia/master-diploma/data/belgiumts/sequences_jpg/Seq01/01/' + img
    obj = {'data': {'image': label_studio_path}, 'predictions': [{'model_version': 'one', 'score': 0.5, 'result': result}]}
    label_studio_anns.append(obj)

In [23]:
with open('./data/belgiumts/sequences_jpg/label_studio_seq01_01.json', 'w') as f:
    json.dump(label_studio_anns, f)

In [4]:
model = YOLO('./models/train4/weights/best.pt')
# model = YOLO('./models/kursova.pt')

In [76]:
images = [
    '/home/viktor/Downloads/dfg-cat-webpage-1.jpg',
    '/home/viktor/Downloads/dfg-cat-webpage-2.jpg',
    '/home/viktor/Downloads/dfg-cat-webpage-3.jpg',
    '/home/viktor/Downloads/dfg-cat-webpage-4.jpg',
    '/home/viktor/Downloads/dfg-cat-webpage-5.jpg',
]

cls_name = 'x-1.1'

for img_path in images:
    result = model.predict(img_path)
    img = result[0].orig_img
    show = False
    for (cls, xyxy) in zip(result[0].boxes.cls, result[0].boxes.xyxy):
        if model.names[int(cls)].lower() == cls_name:
            x1, y1, x2, y2 = map(int, xyxy)
            cv.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            show = True
    if show:
        cv.imshow('Result', img)
        key = cv.waitKey(0)
        cv.destroyAllWindows()
        if key != ord('c'):
            break


image 1/1 /home/viktor/Downloads/dfg-cat-webpage-1.jpg: 640x640 1 I-1.1, 1 I-10, 1 I-13, 1 I-13.1, 1 I-14, 1 I-15, 1 I-16, 1 I-20, 1 I-28, 1 I-29.1, 2 I-4s, 1 I-5.1, 1 I-8, 1 I-9, 1 II-10.1, 1 II-18, 1 II-19-4, 1 II-28, 1 II-39, 1 II-43, 1 II-6, 1 II-7, 1 II-7.1, 1 III-12, 1 III-14, 1 III-14.1, 1 III-15, 1 III-25.1, 1 III-29-30, 1 III-29-40, 1 III-35, 1 III-43, 1 III-45, 1 III-46, 1 III-47, 1 III-5, 1 III-50, 1 III-54, 1 III-59, 1 III-6, 1 III-78, 1 III-86-1, 1 III-86-2, 1 IV-13-3, 1 IV-13-4, 1 IV-13-5, 1 IV-13-6, 1 IV-18, 1577.2ms
Speed: 4.1ms preprocess, 1577.2ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/viktor/Downloads/dfg-cat-webpage-2.jpg: 640x640 1 I-1, 1 I-10, 1 I-11, 1 I-13.1, 1 I-17, 1 I-25, 1 I-29, 1 I-29.1, 1 I-30, 1 I-36, 1 I-37, 1 I-5, 1 I-5.2, 1 I-9, 1 II-17, 1 II-18, 1 II-19-4, 1 II-3, 1 II-32, 1 II-33, 1 II-8, 1 III-107-1, 1 III-107-2, 1 III-107.1-1, 1 III-107.2-1, 1 III-107.2-2, 1 III-120-1, 1 III-15, 1 III-203-2, 1 III-29-30, 

In [15]:
mapped_classes_from_belgiumts_to_dfg = {}

img_folder = './data/belgiumts/img'

for filename in sorted(os.listdir(img_folder)):
    img_path = os.path.join(img_folder, filename)

    # Skip if not a file
    if not os.path.isfile(img_path):
        continue

    # Read image
    img = cv.imread(img_path)
    if img is None:
        continue

    # Add padding (100 pixels on all sides)
    padding = 100
    padded_img = cv.copyMakeBorder(
        img,
        padding, padding, padding, padding,
        cv.BORDER_CONSTANT,
        value=[0, 0, 0]
    )

    # Run detection
    results = model.predict(padded_img, verbose=False)

    # Print detected classes
    if len(results[0].boxes) > 0:
        detected_classes = [model.names[int(cls)]
                            for cls in results[0].boxes.cls]
        mapped_classes_from_belgiumts_to_dfg[filename.split(
            '.')[0].lower()] = detected_classes
        print(f"{filename}: {', '.join(detected_classes)}")
    else:
        mapped_classes_from_belgiumts_to_dfg[filename.split('.')[
            0].lower()] = []
        print(f"{filename}: No detections")

A11.png: No detections
A13.png: I-8
A14.png: I-8
A15.png: No detections
A17.png: No detections
A19.png: I-13.1
A1A.png: No detections
A1B.png: No detections
A1C.png: I-2
A1D.png: I-2.1
A21.png: No detections
A23.png: No detections
A25.png: I-32
A27.png: No detections
A29.png: I-8
A3.png: I-3, I-13.1
A31.png: No detections
A33.png: I-20
A35.png: No detections
A37.png: No detections
A39.png: No detections
A41.png: No detections
A43.png: I-37
A49.png: I-8
A5.png: No detections
A51.png: I-25, III-120
A7A.png: No detections
A7B.png: I-5.2
A7C.png: I-5.1
A9.png: No detections
B1.png: No detections
B11.png: III-3
B13.png: No detections
B15A.png: No detections
B17.png: No detections
B19.png: II-33
B21.png: III-1
B3.png: No detections
B5.png: II-2
B7.png: No detections
B9.png: III-3
C1.png: No detections
C11.png: No detections
C13.png: No detections
C15.png: No detections
C17.png: No detections
C19.png: II-33
C21.png: No detections
C22.png: No detections
C23.png: No detections
C24a.png: No dete

In [3]:
with open('./data/belgiumts/sequences_jpg/seq02_01_label_studio_out_coco.json', 'r') as f:
    label_studio_coco = json.load(f)

label_studio_coco.keys()

dict_keys(['images', 'categories', 'annotations', 'info'])

In [4]:
for img in label_studio_coco['images']:
    file = img['file_name'].split('/')[-1]
    new_path = 'Seq02/01/' + file
    img['file_name'] = new_path

In [5]:
label_studio_coco['images'][0]

{'width': 1628,
 'height': 1236,
 'id': 0,
 'file_name': 'Seq02/01/image.000800.jpg'}

In [6]:
with open('./data/belgiumts/sequences_jpg/seq02_01_label_studio_out_coco.json', 'w') as f:
    json.dump(label_studio_coco, f)

In [53]:
with open('./data/belgiumts/sequences_jpg/seq02_01_mapping.txt', 'r') as f:
    lines = f.readlines()

mapping = {}
for line in lines:
    if line.strip():
        key, value = line.strip().split(':')
        mapping[key.strip()] = value.strip()

In [54]:
with open('./data/belgiumts/sequences_jpg/seq02_01_track.json', 'r') as f:
    yolo_annotations = json.load(f)

yolo_annotations_by_img = {}
for ann in yolo_annotations:
    image = ann['image'].replace('.jp2', '.jpg')
    if image not in yolo_annotations_by_img:
        yolo_annotations_by_img[image] = []
    yolo_annotations_by_img[image].append(ann)

In [55]:
with open('./data/belgiumts/sequences_jpg/seq02_01_reconstructed.json', 'r') as f:
    reconstructed_annotations = json.load(f)

reconstructed_annotations_by_img = {}
for ann in reconstructed_annotations:
    if ann['class'] not in mapping:
        continue
    ann['class'] = mapping[ann['class']]
    image = ann['image'].replace('.jp2', '.jpg')
    if image not in reconstructed_annotations_by_img:
        reconstructed_annotations_by_img[image] = []
    reconstructed_annotations_by_img[image].append(ann)

In [65]:
label_studio_anns = []
id = 1
# for img, anns in annotations_by_img.items():
for img in sorted(os.listdir('./data/belgiumts/sequences_jpg/Seq02/01/')):
    yolo_anns = yolo_annotations_by_img.get(img, [])
    yolo_result = []
    for ann in yolo_anns:
        x1 = ann['bbox'][0] - ann['bbox'][2] / 2
        y1 = ann['bbox'][1] - ann['bbox'][3] / 2
        yolo_result.append({
            'id': f'yoloresult{id}',
            'type': 'rectanglelabels',
            'from_name': 'label',
            'to_name': 'image',
            'original_width': 1628,
            'original_height': 1236,
            'image_rotation': 0,
            'value': {
                'rotation': 0,
                'x': x1 / 1628 * 100,
                'y': y1 / 1236 * 100,
                'width': ann['bbox'][2] / 1628 * 100,
                'height': ann['bbox'][3] / 1236 * 100,
                'rectanglelabels': [ann['class']]
            }
        })
        id += 1

    reconstructed_anns = reconstructed_annotations_by_img.get(img, [])
    reconstructed_result = []
    for ann in reconstructed_anns:
        x1 = ann['bbox'][0] - ann['bbox'][2] / 2
        y1 = ann['bbox'][1] - ann['bbox'][3] / 2
        reconstructed_result.append({
            'id': f'reconstructedresult{id}',
            'type': 'rectanglelabels',
            'from_name': 'label',
            'to_name': 'image',
            'original_width': 1628,
            'original_height': 1236,
            'image_rotation': 0,
            'value': {
                'rotation': 0,
                'x': x1 / 1628 * 100,
                'y': y1 / 1236 * 100,
                'width': ann['bbox'][2] / 1628 * 100,
                'height': ann['bbox'][3] / 1236 * 100,
                'rectanglelabels': [ann['class']]
            }
        })
        id += 1

    label_studio_path = 'http://localhost:8081/Seq02/01/' + img

    obj = {'data': {'image': label_studio_path}, 'predictions': [
        {'model_version': 'yolo', 'score': 0.5, 'result': yolo_result},
        {'model_version': 'reconstructed', 'score': 0.5, 'result': reconstructed_result},
        {'model_version': 'yolo_plus_reconstructed', 'score': 0.5, 'result': yolo_result + reconstructed_result}]}

    label_studio_anns.append(obj)

In [66]:
with open('./data/belgiumts/sequences_jpg/label_studio_seq02_01_localserve.json', 'w') as f:
    json.dump(label_studio_anns, f)

In [59]:
with open('./data/belgiumts/sequences_jpg/label_studio_seq01_01.json', 'r') as f:
    label_studio_seq01_01 = json.load(f)

with open('./data/belgiumts/sequences_jpg/label_studio_seq02_01.json', 'r') as f:
    label_studio_seq02_01 = json.load(f)